In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score
from sklearn.metrics import roc_curve, confusion_matrix, auc, recall_score, precision_recall_curve
from sklearn.metrics import brier_score_loss, hamming_loss
import numpy as np
import pandas as pd

In [3]:
import pickle
with open('balanced_data.pkl', 'rb') as b:
    data = pickle.load(b)

X_res_sk = data['X_res_stroke']
y_res_sk = data['y_res_stroke']
X_res_db = data['X_res_diab']
y_res_db = data['y_res_diab']
X_res_cn = data['X_res_can']
y_res_cn = data['y_res_can']       
X_res_ha = data['X_res_ha']
y_res_ha = data['y_res_ha']    

X_val = data['X_val'] 
y_val = data['y_val']   

X_test = data['X_test']
y_test = data['y_test']

X_bal=data['X_bal']
y_bal=data['y_bal']



In [4]:
y_test=y_test.rename(columns=
              {'heart_attack' : 'HA'})
y_bal=y_bal.rename(columns=
              {'heart_attack' : 'HA'})
y_val=y_val.rename(columns=
              {'heart_attack' : 'HA'})



In [ ]:
param_grid = {
    'n_estimator' : [100,300,500],
    'max_depth' : [None, 5, 10, 15]
}

def multilabel_prob(model, X):
    return np.stack([p[:,1] for p in model.predict_proba(X)])

label_name = y_bal.columns.tolist()

best_score  = np.inf 
best_params = None
best_rf     = None

for n_est in param_grid['n_estimator']:
    for max_depth in param_grid['max_depth']:
        
        rf = RandomForestClassifier(
            n_estimators= n_est,
            max_depth=max_depth,
            criterion= 'entropy',
            class_weight='balanced',
            random_state=1234,
            max_features='sqrt'
        )
        
        rf.fit(X_bal, y_bal)
        
        y_val_prob = multilabel_prob(rf, X_val)
        y_val_pred = (y_val_prob >= 0.5).astype(int).T
        
        misc = np.mean([
                np.mean(y_val[col].values != y_val_pred[:, i])
                for i, col in enumerate(label_name)
            ])
 
        if misc < best_score:
            best_score = misc
            best_params = {
                'n_estimator' : n_est,
                'max_depth' : max_depth
            }
            best_rf = rf
            
print(f"Best params:{best_params}")
print(f"Best misclassification:{best_score:.4f}")          

rf_model = best_rf 



Best params:             {'n_estimator': 300, 'max_depth': None}
Best misclassification:  0.0829


In [ ]:
y_test_prob = multilabel_prob(best_rf, X_test)
y_test_pred = (y_test_prob >= 0.5).astype(int).T


results_rf = {}

for i, col in enumerate(y_test.columns):
    y_test_pred_i = y_test_pred[:, i]
    y_test_prob_i = y_test_prob[i, :] 
    results_rf[col] = {
        'f1' : f1_score(y_test[col], y_test_pred_i),
        'misc' : (y_test_pred_i != y_test[col].values).mean(),
        'precision' : precision_score(y_test[col], y_test_pred_i),
        'confusion_matrix' : confusion_matrix( y_test[col].values, y_test_pred_i),
        'sens' : recall_score(y_test[col].values, y_test_pred_i),
        'auc-pr' : average_precision_score(y_test[col].values, y_test_prob_i),
        'auc' : roc_auc_score(y_test[col], y_test_prob_i),
        "brier": brier_score_loss(y_test[col].values,y_test_prob_i),
        'thresholds' : 0.50  
    }

y_pred_multi = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_test_prob[i, :] >= 0.5).astype(int)

macro_f1 = f1_score(y_test.values, y_pred_multi, average='macro')
hamming = hamming_loss(y_test.values, y_pred_multi)

results_rf["MacroF1"] = macro_f1
results_rf["Hamming_ml"] = hamming

brier_mean = np.mean([
    brier_score_loss(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])
macro_auc = np.mean([
    roc_auc_score(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])
macro_auc_pr = np.mean([
    average_precision_score(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])

results_rf["Brier_mean"] = brier_mean
results_rf["MacroAUC"] = macro_auc
results_rf["MacroAUC_PR"] = macro_auc_pr

print(pd.DataFrame({k: v for k, v in results_rf.items() if k != "confusion_matrix"}).T)

In [ ]:

def youden_threshold(y_true_col, y_prob_col):
    fpr, tpr, thresholds = roc_curve(y_true_col, y_prob_col)
    return thresholds[np.argmax(tpr - fpr)]
    
    
param_grid = {
    'n_estimator' : [100,300,500],
    'max_depth' : [None, 5, 10, 15]
}

def multilabel_prob(model, X):
    return np.stack([p[:,1] for p in model.predict_proba(X)])

label_name = y_bal.columns.tolist()

best_score  = np.inf 
best_params = None
best_rf     = None
best_thr     = None

for n_est in param_grid['n_estimator']:
    for max_depth in param_grid['max_depth']:
        
        rf = RandomForestClassifier(
            n_estimators= n_est,
            max_depth=max_depth,
            criterion= 'entropy',
            class_weight='balanced',
            random_state=1234,
            max_features='sqrt'
        )
        
        rf.fit(X_bal, y_bal)
        
        y_val_prob = multilabel_prob(rf, X_val)
        
       
        errors = []
        thresholds = {}

        # Youden and misclassification thresholds on validation
        for i, col in enumerate(y_val.columns):
            thr = youden_threshold(y_val[col].values, y_val_prob[i, :])
            thresholds[col] = thr
            y_pred_i = (y_val_prob[i, :] >= thr).astype(int)
            errors.append(
                np.mean(y_val[col].values != y_pred_i)
            )
            
        misclass = np.mean(errors)
        
        if misclass < best_score:
            best_score  = misclass
            best_params = {
                'n_estimator': n_est,
                'max_depth': max_depth
            }
            best_rf  = rf
            best_thr = thresholds
            
print(f"Best params:             {best_params}")
print(f"Best misclassification:  {best_score:.4f}")          

rf_model = best_rf 

Best params:             {'n_estimator': 500, 'max_depth': 10}
Best misclassification:  0.3001


In [ ]:
y_test_prob = multilabel_prob(best_rf, X_test)

results_youden = {}

for i, col in enumerate(y_test.columns):
    thr = best_thr[col]
    y_prob_i = y_test_prob[i,:]
    y_pred_i = (y_test_prob[i,:] >= thr).astype(int)
        
    results_youden[col] = {
        "f1": f1_score(y_test[col], y_pred_i),
        'misc' : (y_pred_i != y_test[col].values).mean(),
        'precision' : precision_score(y_test[col], y_pred_i),
        'confusion_matrix' : confusion_matrix( y_test[col].values, y_pred_i),
        'sens' : recall_score(y_test[col].values, y_pred_i),
        'auc-pr' : average_precision_score(y_test[col].values, y_prob_i),  # average precision instead of AUC: the x axis is not monotonic
        'auc' : roc_auc_score(y_test[col].values, y_prob_i),
        "brier": brier_score_loss(y_test[col].values, y_prob_i),
        'thresholds' : best_thr[col]
    }

y_pred_multi = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_test_prob[i,:] >= best_thr[col]).astype(int)

macro_f1 = f1_score(y_test.values, y_pred_multi, average='macro')
hamming = hamming_loss(y_test.values, y_pred_multi)

results_youden["MacroF1"] = macro_f1
results_youden["Hamming"] = hamming

brier_mean = np.mean([
    brier_score_loss(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])
macro_auc = np.mean([
    roc_auc_score(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])
macro_auc_pr = np.mean([
    average_precision_score(y_test[col].values, y_test_prob[i, :])
    for i, col in enumerate(y_test.columns)
])

results_youden["Brier_mean"] = brier_mean
results_youden["MacroAUC"] = macro_auc
results_youden["MacroAUC_PR"] = macro_auc_pr


print(pd.DataFrame({k: v for k, v in results_youden.items() if k != "confusion_matrix"}).T)

                   f1      misc precision            confusion_matrix  \
diabetes     0.431208  0.294079  0.297798  [[2741, 1212], [144, 514]]   
stroke       0.114708  0.338105   0.06285   [[2951, 1506], [53, 101]]   
HA           0.313901  0.331815  0.208086  [[2731, 1332], [198, 350]]   
cancer       0.097436  0.229018  0.058582    [[3498, 916], [140, 57]]   
MacroF1      0.239313  0.239313  0.239313                    0.239313   
Hamming      0.298254  0.298254  0.298254                    0.298254   
Brier_mean   0.150113  0.150113  0.150113                    0.150113   
MacroAUC     0.703657  0.703657  0.703657                    0.703657   
MacroAUC_PR  0.208369  0.208369  0.208369                    0.208369   

                 sens    auc-pr       auc     brier thresholds  
diabetes     0.781155   0.46489  0.813766  0.182931   0.472692  
stroke       0.655844  0.069454  0.706185  0.107317   0.334448  
HA           0.638686  0.243981  0.720619  0.217646   0.509316  
cancer   

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    quantiles = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    ece = 0
    n = len(y_prob)
    for b in range(n_bins):
        mask = (y_prob >= quantiles[b]) & (y_prob < quantiles[b+1])
        if mask.sum() > 0:
            acc = y_true[mask].mean()   
            conf = y_prob[mask].mean()  
            ece += mask.mean() * abs(acc - conf)
    return ece


for i, col in enumerate(y_test.columns):
    ece = expected_calibration_error(
        y_test[col].values, 
        y_test_prob[i, :]
    )
    print(f'{col}: ECE = {ece:.4f}')
    results_youden[col]["ece"] = ece

diabetes: ECE = 0.2828
stroke: ECE = 0.2701
HA: ECE = 0.3444
cancer: ECE = 0.2266


In [10]:
import pickle

with open("results_rf.pkl", "wb") as f:
    pickle.dump(results_youden, f)

In [ ]:

def recall_threshold(y_true_col, y_prob_col, recall_target=0.95, grid=200):
    thresholds = np.linspace(0, 1, grid)
    best_thr = 0.0

    for thr in thresholds:
        y_pred = (y_prob_col >= thr).astype(int)
        rec = recall_score(y_true_col, y_pred, zero_division=0)

        if rec >= recall_target:
            best_thr = thr  

    return best_thr


RECALL_TARGET = 0.90

best_score = np.inf
best_params_recall = None
best_rf_recall = None
best_thr_recall = None

for n_est in param_grid['n_estimator']:
    for max_depth in param_grid['max_depth']:

        rf = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=max_depth,
            criterion='entropy',
            class_weight='balanced',
            random_state=1234,
            max_features='sqrt'
        )

        rf.fit(X_bal, y_bal)
        y_val_prob = multilabel_prob(rf, X_val)

        errors = []
        thresholds = {}

        for i, col in enumerate(y_val.columns):
            thr = recall_threshold(
                y_val[col].values,
                y_val_prob[i, :],
                recall_target=RECALL_TARGET
            )
            thresholds[col] = thr
            y_pred_i = (y_val_prob[i, :] >= thr).astype(int)
            errors.append((y_pred_i != y_val[col].values).mean())

        misclass = np.mean(errors)

        if misclass < best_score:
            best_score = misclass
            best_params_recall = {
                'n_estimator': n_est,
                'max_depth': max_depth
            }
            best_rf_recall = rf
            best_thr_recall = thresholds

print(f"Best params recall:             {best_params_recall}")
print(f"Best misclassification @recall: {best_score:.4f}")

y_test_prob_recall = multilabel_prob(best_rf_recall, X_test)
results_recall_rf = {}

for i, col in enumerate(y_test.columns):
    thr = best_thr_recall[col]
    y_prob_i = y_test_prob_recall[i, :]
    y_pred_i = (y_prob_i >= thr).astype(int)

    results_recall_rf[col] = {
        'f1': f1_score(y_test[col].values, y_pred_i),
        'misc': (y_pred_i != y_test[col].values).mean(),
        'precision': precision_score(y_test[col].values, y_pred_i, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test[col].values, y_pred_i),
        'sens': recall_score(y_test[col].values, y_pred_i, zero_division=0),
        'auc-pr': average_precision_score(y_test[col].values, y_prob_i),
        'auc': roc_auc_score(y_test[col].values, y_prob_i),
        'brier': brier_score_loss(y_test[col].values, y_prob_i),
        'thresholds': thr
    }

y_pred_multi = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_test_prob_recall[i, :] >= best_thr_recall[col]).astype(int)

results_recall_rf["MacroF1"] = f1_score(y_test.values, y_pred_multi, average='macro')
results_recall_rf["Hamming"] = hamming_loss(y_test.values, y_pred_multi)
results_recall_rf["Brier_mean"] = np.mean([
    brier_score_loss(y_test[col].values, y_test_prob_recall[i, :])
    for i, col in enumerate(y_test.columns)
])
results_recall_rf["MacroAUC"] = np.mean([
    roc_auc_score(y_test[col].values, y_test_prob_recall[i, :])
    for i, col in enumerate(y_test.columns)
])
results_recall_rf["MacroAUC_PR"] = np.mean([
    average_precision_score(y_test[col].values, y_test_prob_recall[i, :])
    for i, col in enumerate(y_test.columns)
])

print(pd.DataFrame({k: v for k, v in results_recall_rf.items() if k != "confusion_matrix"}).T)

Best params recall:             {'n_estimator': 500, 'max_depth': None}
Best misclassification @recall: 0.5927
                   f1      misc precision           confusion_matrix  \
diabetes     0.379818  0.429191  0.239242  [[2026, 1927], [52, 606]]   
stroke       0.087702  0.600087  0.046197  [[1711, 2746], [21, 133]]   
HA           0.266808  0.598352  0.156143  [[1350, 2713], [46, 502]]   
cancer       0.094418  0.752982  0.049766   [[958, 3456], [16, 181]]   
MacroF1      0.207187  0.207187  0.207187                   0.207187   
Hamming      0.595153  0.595153  0.595153                   0.595153   
Brier_mean   0.066738  0.066738  0.066738                   0.066738   
MacroAUC     0.725598  0.725598  0.725598                   0.725598   
MacroAUC_PR  0.226758  0.226758  0.226758                   0.226758   

                 sens    auc-pr       auc     brier thresholds  
diabetes     0.920973  0.499236  0.836798  0.096612   0.095477  
stroke       0.863636  0.085935  0.711

In [12]:

COST_FN = 10
COST_FP = 1

def compute_cost(y_true_col, y_pred_col, cost_fn=10, cost_fp=1):
    tn, fp, fn, tp = confusion_matrix(
        y_true_col,
        y_pred_col,
        labels=[0, 1]
    ).ravel()
    return cost_fn * fn + cost_fp * fp


def cost_threshold(y_true_col, y_prob_col, cost_fn=10, cost_fp=1, grid=100):
    thresholds = np.linspace(0.01, 0.99, grid)
    best_thr = 0.5
    best_cost = np.inf

    for thr in thresholds:
        y_pred = (y_prob_col >= thr).astype(int)
        cost = compute_cost(y_true_col, y_pred, cost_fn, cost_fp)

        if cost < best_cost:
            best_cost = cost
            best_thr = thr

    return best_thr


best_cost = np.inf
best_params_cost = None
best_rf_cost = None
best_thr_cost = None

for n_est in param_grid['n_estimator']:
    for max_depth in param_grid['max_depth']:

        rf = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=max_depth,
            criterion='entropy',
            class_weight='balanced',
            random_state=1234,
            max_features='sqrt'
        )

        rf.fit(X_bal, y_bal)
        y_val_prob = multilabel_prob(rf, X_val)

        costs = []
        thresholds = {}

        for i, col in enumerate(y_val.columns):
            thr = cost_threshold(
                y_val[col].values,
                y_val_prob[i, :],
                cost_fn=COST_FN,
                cost_fp=COST_FP
            )
            thresholds[col] = thr
            y_pred_i = (y_val_prob[i, :] >= thr).astype(int)
            costs.append(
                compute_cost(y_val[col].values, y_pred_i, COST_FN, COST_FP)
            )

        mean_cost = np.mean(costs)

        if mean_cost < best_cost:
            best_cost = mean_cost
            best_params_cost = {
                'n_estimator': n_est,
                'max_depth': max_depth
            }
            best_rf_cost = rf
            best_thr_cost = thresholds

print(f"Best params cost:     {best_params_cost}")
print(f"Best validation cost: {best_cost:.4f}")

y_test_prob_cost = multilabel_prob(best_rf_cost, X_test)
results_cost_rf = {}

for i, col in enumerate(y_test.columns):
    thr = best_thr_cost[col]
    y_prob_i = y_test_prob_cost[i, :]
    y_pred_i = (y_prob_i >= thr).astype(int)

    results_cost_rf[col] = {
        'f1': f1_score(y_test[col].values, y_pred_i),
        'misc': (y_pred_i != y_test[col].values).mean(),
        'precision': precision_score(y_test[col].values, y_pred_i, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test[col].values, y_pred_i),
        'sens': recall_score(y_test[col].values, y_pred_i, zero_division=0),
        'auc-pr': average_precision_score(y_test[col].values, y_prob_i),
        'auc': roc_auc_score(y_test[col].values, y_prob_i),
        'brier': brier_score_loss(y_test[col].values, y_prob_i),
        'cost': compute_cost(y_test[col].values, y_pred_i, COST_FN, COST_FP),
        'thresholds': thr
    }

y_pred_multi = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_test_prob_cost[i, :] >= best_thr_cost[col]).astype(int)

results_cost_rf["MacroF1"] = f1_score(y_test.values, y_pred_multi, average='macro')

results_cost_rf["Hamming"] = hamming_loss(y_test.values, y_pred_multi)

results_cost_rf["Brier_mean"] = np.mean([
    brier_score_loss(y_test[col].values, y_test_prob_cost[i, :])
    for i, col in enumerate(y_test.columns)
])
results_cost_rf["MacroAUC"] = np.mean([
    roc_auc_score(y_test[col].values, y_test_prob_cost[i, :])
    for i, col in enumerate(y_test.columns)
])
results_cost_rf["MacroAUC_PR"] = np.mean([
    average_precision_score(y_test[col].values, y_test_prob_cost[i, :])
    for i, col in enumerate(y_test.columns)
])
results_cost_rf["Mean_cost"] = np.mean([
    results_cost_rf[col]["cost"]
    for col in y_test.columns
])

print(pd.DataFrame({k: v for k, v in results_cost_rf.items() if k != "confusion_matrix"}).T)



Best params cost:     {'n_estimator': 300, 'max_depth': 10}
Best validation cost: 2215.5000
                   f1      misc precision            confusion_matrix  \
diabetes      0.39749  0.374756  0.257919   [[2313, 1640], [88, 570]]   
stroke       0.105066  0.103448  0.073879    [[4106, 351], [126, 28]]   
HA            0.31489   0.40013  0.197669  [[2342, 1721], [124, 424]]   
cancer            0.0  0.044025       0.0       [[4408, 6], [197, 0]]   
MacroF1      0.204361  0.204361  0.204361                    0.204361   
Hamming       0.23059   0.23059   0.23059                     0.23059   
Brier_mean   0.149066  0.149066  0.149066                    0.149066   
MacroAUC     0.704723  0.704723  0.704723                    0.704723   
MacroAUC_PR  0.208268  0.208268  0.208268                    0.208268   
Mean_cost      2267.0    2267.0    2267.0                      2267.0   

                 sens    auc-pr       auc     brier      cost thresholds  
diabetes     0.866261  0.4623

In [14]:
results_rf_thresholds = {
    "fixed_050": results_rf,
    "youden": results_youden,
    "recall": results_recall_rf,
    "cost": results_cost_rf
}

with open("results_rf_thresholds.pkl", "wb") as f:
    pickle.dump(results_rf_thresholds, f)


In [15]:
with open("results_rf_youden.pkl", "wb") as f:
    pickle.dump(results_youden, f)